The other day, I opened Leetcode for the first time in a year. I took out a random medium question, and all of a sudden, my eyes glazed over. It was pretty obvious how to solve it (dynamic programming, who would've thought?) but I just couldn't understand what that solution would look like on paper. After 20 minutes of struggling, I gave up and looked at a solution. And to be honest, [the explanation I found](https://leetcode.com/problems/edit-distance/solutions/159295/python-solutions-and-intuition-by-anders-amxq/?envType=problem-list-v2&envId=dynamic-programming) did much more to explain dynamic programming than my [CS341](https://uwflow.com/course/cs341) professors ever did.

This article is my attempt to (re)understand of dynamic programming. Hopefully it helps you understand it better too, but I'm writing this more so I don't forget the intuition I gained in that moment. If you're a recruiter, please go easy on me, I haven't opened Leetcode since writing this 😅

*Note: this article assumes knowledge of basic computer science principles, such as recursion, common data structures, and time conplexity.*

### What is Dynamic Programming?

To put it the way those professors did: dynamic programming (DP) is the act of using the results of subproblems to solve larger ones, without solving the smaller problems over and over again. We perform DP by memoizing the results of those smaller problems, so that we only calculate them once.

Why is it important that we only calculate those intermediate results once? That's because we use DP to solve problems with an exponential amount of computations, or that are heavily recusrive. The key to DP's effectiveness is recognizing which calculations are repetitive, and then refusing to repeat them, leading to an $O(n)$ or $O(n^2)$ solution.

Each DP solution is based on your response to the following three questions:
- what subproblem am I solving?
- how do I index the result to the subproblem (i.e. how do I distinguish each subproblem from the others)?
- what action do I take to go to the next subproblem?

### A Quick Example You've Surely Seen Before

The obvious example of DP's effectiveness is computing the Fibonacci sequence. This is where my textbook came up with an iterative approach involving an array. But I want to build intuition from the ground up. As I mentioned before, we use DP for heavily recursive problems, and Fibonacci is just that. So let's start with the formula for the $n$ th Fibonacci number:

$$ f(n) = f(n - 1) + f(n - 2), f(1) = 1, f(2) = 1 $$

From this, we can answer our 2 questions:
- what subproblem am I solving? Each Fibonacci number depends on the 2 that came before it. So, the subproblem we are solving is the $i$ th Fibonacci number, $3 < i <= n$.
- how do I index the result to the subproblem? Easy, we use the index $i$.
- what action do I take to go to the next subproblem? We grab the value for $f(i - 1)$ and $f(i - 2)$.

Now we can look at implementing the solution. For a recursive problem like this, it's intuitive to write a recursive function:

In [2]:
def fibonacci(n):
    if n == 1 or n == 2:
        return 1

    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(5)

5

This solution has an obvious flaw: we recalculate the same values many, many times. We can see this if we trace out a function call, even for a small value like 5:

$$ f(5) $$
$$ = f(3) + f(4) $$
$$ = (f(2) + f(1)) + (f(4) + f(3)) $$
$$ = (f(2) + f(1)) + ((f(3) + f(2)) + f(3)) $$
$$ = (f(2) + f(1)) + (((f(2) + f(1)) + f(2)) + (f(2) + f(1))) $$

From this we see that our single calculation actually leads to 7 leaf calculations (given that $f(2)$ and $f(1)$ are base cases). This is a lot of computation that we do not need to do (in fact, an exponential amount). Instead, let's save the results of our subproblems in a dictionary. We can calculate with the results of the subproblem as is, only, we do a cache lookup first.

In [38]:
def fibonacci(n):
    cache = dict()

    def fibonacci_impl(i):
        if cache.get(i):
            return cache[i]

        if i == 1 or i == 2:
            return 1
        
        cache[i] = fibonacci_impl(i - 1) + fibonacci_impl(i - 2)

        return cache[i]
    
    return fibonacci_impl(n)

fibonacci(5)

5

In the above implementation, we perform the lookup before recursing, which prevents the majority of the computation from ever happening. We can see this in the math directly, marking the cache state with $\textcolor{red}{[miss]}$ or $\textcolor{green}{[hit]}$:

$$ f(5) \textcolor{red}{[miss]} $$
$$ = f(3) \textcolor{red}{[miss]} + f(4) $$
$$ = (f(2) \textcolor{red}{[miss]} + f(1) \textcolor{red}{[miss]}) + f(4) $$
$$ = (f(2) + f(1)) + (f(3) \textcolor{green}{[hit]} + f(2) \textcolor{green}{[hit]}) $$

The recursion ends much sooner here, and as a result, this implementation is now $O(n)$ instead of exponential time since we only calculate once per value.

To finally reach the iterative approach shown in most textbooks, we unroll the recursion we had before, and store intermediate results in a table (note the use of a list instead of a dictionary to match the textbook implementation).

In [39]:
def fibonacci(n):
    t = [0 for _ in range(n)]
    t[0] = 1

    if len(t) > 1:
        t[1] = 1

    for i in range(2, n):
        t[i] = t[i - 1] + t[i - 2]
    
    return t[n - 1]

fibonacci(5)

5

In practice, an iterative solution is always the gold standard, because deep recursion can lead to stack overflows in large cases, and iterative solutions are usually easier to follow and maintain. However, they are also the hardest to derive from first principles, as we'll see below.

Let's move on to [a more complicated problem](https://leetcode.com/problems/edit-distance/?envType=problem-list-v2&envId=dynamic-programming), this time the one with the solution I got my inspiration for this article from. 

### A Harder problem

Before carrying on, it's best to read the problem description (linked above) and try to ask the 3 questions I gave above (which I'll do now).

1. what subproblem am I solving?

I am trying to figure out the smallest number of moves to match `word1[i:]` to `word2[j:]`. This means, we are a certain way through `word1`, and a certain way through `word2`, what is the minimum edit distance for the remainder of the string?

Why are we trying to figure that out? [fr tho why]

2. how do I index the result to the subproblem (i.e. how do I distinguish each subproblem from the others)?

We have an index for `word1`, `i`, and an index for `word2`, `j`. That is enough to determine how much of each string we have read.

3. what action do I take to go to the next subproblem?

This is where things get interesting. We have 4 we can possibly take in the problem:
- insert a character
- delete a character
- replace a character
- match a character which is in the same position in both strings, in which case we can move on to the next character in both strings. This action is not documented in the question, and we will need to handle it differently.

How do we represent these actions in code? We need to increment our indices, `i` and/or `j`. 
- for an insertion: we are inserting the character from `word1` into `word2`. This means we are accounting for the current character in `word2` by saying that it was added at that position. This means that we are not accounting for the current character in `word1`, because this is not an action which would transform it (i.e. it is not deleted or replaced, it may still exist in `word2`). Because we are accounting for the character in `word2`, this is the same as moving from subproblem `(i, j)` to `(i, j + 1)`.
- for a deletion: we are effectively saying that we are not taking the current character from `word1`. This means we are not accounting for the current character in `word2`, because we don't know its source (it may have been inserted, or the target of a replacement, previously). Because we are accounting for the character in `word1`, this is the same as moving from subproblem `(i, j)` to `(i + 1, j)`.
- for a replacement: we are saying the current character in `word1` maps to the current character in `word2`. This means we account for the current character in both words, meaning we go from subproblem `(i, j)` to `(i + 1, j + 1)`.

The match case is actually the same the replacement case. We are saying the current character in `word1` maps to the current character in `word2`. This means we account for the current character in both words, so we go from subproblem `(i, j)` to `(i + 1, j + 1)`. But it differs from the replacement case in one significant way. 

We have our scheme for incremeting the indices, but we want to take the path of edits which leads to the smallest edit distance. For the match case, we do not perform any edits, so we increment our indices and continue. For the case where we perform any operation, we need to add one to our count of operations, and we need to ensure we continue on the path which leads to the smallest number of edits. We don't know which this is, so instead of trying to identify the operation, we select the one that returns the smallest distance on the rest of the string. In code, that looks like this:

```
# characters match
if word1[i] == word2[j]:
    return subproblem(i + 1, j + 1)

# characters do not match, make SOME edit
return 1 + min([
    subproblem(i, j + 1),
    subproblem(i + 1, j),
    subproblem(i + 1, j + 1)
])
```

The point is, in the case we need to make an edit, we DO NOT care which edit is being made, because they all count for the same thing. (Also, multiple edit paths on `word1` can yield `word2` with a different edit distance, so we need to consider all of them to get the minimum). So we always pick the one with the smallest distance.

Here's the tl;dr:
- if the current characters match: we move on to consider the next character in both strings.
- if the current characters do not match: we have 3 possible edit paths, and our answer is the one that yeilds the smallest edit distance for the rest of the string, irrespective of whether we are on the path with the smallest edit distance or now.

This should give us enough info to write a naive recursive implementation (which we already wrote the essence of above).

In [ ]:
def min_distance(word1: str, word2: str) -> int:
    if word1 == "":
        return len(word2)
    elif word2 == "":
        return len(word1)    

    def subproblem(i, j):
        if i >= len(word1):
            return max(len(word2[j:]), 0)
        elif j >= len(word2):
            return max(len(word1[i:]), 0)

        if word1[i] == word2[j]:
            return subproblem(i + 1, j + 1)
        
        return 1 + min([
            subproblem(i, j + 1),
            subproblem(i + 1, j),
            subproblem(i + 1, j + 1)
        ])

    return subproblem(0, 0)

print(min_distance("horse", "ros"))
print(min_distance("a", "ab"))
print(min_distance("ab", "a"))

3
1
1


From the above code, the inefficiency is clear: in the edit case, we have 3 branches of recursion. We can use our indices `(i, j)` to represent solutions in a cache, *because the solution for each subproblem is the minimum based only on the part of the string that is remaining*, i.e. the data needed to solve that subproblem is *contained within the subproblem*. This point is important, and we will get back to it, because it's no always true. Here's the implementation with caching:

In [ ]:
def min_distance(word1: str, word2: str) -> int:
    if word1 == "":
        return len(word2)
    elif word2 == "":
        return len(word1)    

    subproblem_result_cache = dict()

    def subproblem(i, j):
        if i >= len(word1):
            return max(len(word2[j:]), 0)
        elif j >= len(word2):
            return max(len(word1[i:]), 0)
        
        if subproblem_result_cache.get((i, j)):
            return subproblem_result_cache[(i, j)]

        if word1[i] == word2[j]:
            result = subproblem(i + 1, j + 1)
            subproblem_result_cache[(i, j)] = result
            return result
        
        result = 1 + min([
            subproblem(i, j + 1),
            subproblem(i + 1, j),
            subproblem(i + 1, j + 1)
        ])

        subproblem_result_cache[(i, j)]  = result
        return result

    return subproblem(0, 0)

print(min_distance("horse", "ros"))
print(min_distance("a", "ab"))
print(min_distance("ab", "a"))

3
1
1


Converting this to an iterative approach is trickier than with fibonacci: how do we get an iterative approach if the result of a subproblem depends on the result of another subproblem solved in the future? We cannot use the subproblem definition from before, but the logic of going from subproblem stays the same. We just need to think "bottom-up" instead of "top-down". Instead of incrementing the indices based on the edit, we can instead fetch previous results and append to them based on the case. We do this by reversing the update rule we had before:
- on the insertion case, we take from `(i, j - 1)`
- on the deletion case, we take from `(i - 1, j)`
- on the match/replace case, we take from `(i - 1, j - 1)`

How has our subproblem changed? Instead of asking "what is the minimum edit distance for the remainder of the string?" I can now ask "what is the minimum edit distance for the parts of the strings I have seen already?", then add 1 if an edit is made on the current character, or 0 otherwise. The full implementation is below:

In [ ]:
def min_distance(word1: str, word2: str) -> int:
    if word1 == "":
        return len(word2)
    elif word2 == "":
        return len(word1)

    t = [[0 for _ in word2] for _ in word1]

    for i in range(len(word1)):
        for j in range(len(word2)):
            if word1[i] == word2[j]:
                t[i][j] = t[i - 1][j - 1] if i > 0 and j > 0 else 0
            else:
                t[i][j] = 1 + min([
                    t[i][j - 1] if j > 0 else 0,
                    t[i - 1][j] if i > 0 else 0,
                    t[i - 1][j - 1] if i > 0 and j > 0 else 0
                ])
    
    return t[-1][-1]

print(min_distance("horse", "ros"))
print(min_distance("a", "ab"))
print(min_distance("ab", "a"))

3
1
1


### When our assumptions don't hold

[This next problem](https://leetcode.com/problems/paint-house-iv/?envType=problem-list-v2&envId=dynamic-programming) seems easy on the surface, but there's one key that makes it more difficult than the previous one. Let's answer our questions and start to build a solution:

1. what subproblem am I solving?

We want the minimum cost to make all rows up to the ith row beautiful.

2. how do I index the result to the subproblem (i.e. how do I distinguish each subproblem from the others)?

We can index based on i (row we've reached) and j (house color we chose).

3. what action do I take to go to the next subproblem?

We can select a color for the current house, eliminating that option for the next house and the one equidistant from the center to the current house.

This gives us a pretty simple recursive implementation, which you can see below. Each time we paint a house, we can record the color we pick and try each of 2 decisions (the remaining 2 colors) for the other houses.

In [ ]:
from typing import List


def minCost(n: int, cost: List[List[int]]) -> int:
    def subproblem(i, j, t, decisions):
        if i == len(cost):
            return t
        
        def equidistant_house_color_valid(k, l):
            return (len(decisions) > n - k - 1 and decisions[n - k - 1] != l) or len(decisions) <= n - k - 1
        
        r1 = float("inf")

        if j == 0 and equidistant_house_color_valid(i, 0):
            r1 = min([
                subproblem(i + 1, 1, t + cost[i][0], decisions + [0]),
                subproblem(i + 1, 2, t + cost[i][0], decisions + [0]),
            ])
        elif j == 1 and equidistant_house_color_valid(i, 1):
            r1 =  min([
                subproblem(i + 1, 0, t + cost[i][1], decisions + [1]),
                subproblem(i + 1, 2, t + cost[i][1], decisions + [1]),
            ])
        elif j == 2 and equidistant_house_color_valid(i, 2):
            r1 =  min([
                subproblem(i + 1, 0, t + cost[i][2], decisions + [2]),
                subproblem(i + 1, 1, t + cost[i][2], decisions + [2]),
            ])
        
        return r1
    
    return min([
        subproblem(0, 0, 0, []),
        subproblem(0, 1, 0, []),
        subproblem(0, 2, 0, []),
    ])

print(minCost(4, [[3,5,7],[6,2,9],[4,8,1],[7,3,5]]))

9


What happens if we naively try to cache the results here?

In [46]:
# for anyone that glossed over the text from before, this solution DOES NOT WORK

def minCost(n: int, cost: List[List[int]]) -> int:
    cache = dict()

    def subproblem(i, j, t, decisions):
        if i == len(cost):
            cache[(i, j)] = t
            return t

        if cache.get((i, j)):
            return cache[(i, j)]
        
        def equidistant_house_color_valid(k, l):
            return (len(decisions) > n - k - 1 and decisions[n - k - 1] != l) or len(decisions) <= n - k - 1
        
        r1 = float("inf")

        if j == 0 and equidistant_house_color_valid(i, 0):
            r1 = min([
                subproblem(i + 1, 1, t + cost[i][0], decisions + [0]),
                subproblem(i + 1, 2, t + cost[i][0], decisions + [0]),
            ])
        elif j == 1 and equidistant_house_color_valid(i, 1):
            r1 =  min([
                subproblem(i + 1, 0, t + cost[i][1], decisions + [1]),
                subproblem(i + 1, 2, t + cost[i][1], decisions + [1]),
            ])
        elif j == 2 and equidistant_house_color_valid(i, 2):
            r1 =  min([
                subproblem(i + 1, 0, t + cost[i][2], decisions + [2]),
                subproblem(i + 1, 1, t + cost[i][2], decisions + [2]),
            ])
        
        cache[(i, j)] = r1
        return r1
    
    return min([
        subproblem(0, 0, 0, []),
        subproblem(0, 1, 0, []),
        subproblem(0, 2, 0, []),
    ])

print(minCost(4, [[3,5,7],[6,2,9],[4,8,1],[7,3,5]]))

12


This code is pretty much always wrong. Why? Remember the key assumption we used in the last problem:

*"The data needed to solve that subproblem is contained within the subproblem"*

In this one, the equidistance constraint means that with our current subproblem, the data is not contained within the subproblem, because the decision we make is made both based on the previous result and the one for the equidistant house (if it exists), and influences the immediate house after and the equidistant one (if its color hasn't been decided yet). If we cache the result from one path, it may not be the result which guarantees a global minimum, which in turn will throw off the entire calculation. We can see that with one of the default tests from Leetcode: `[[3,5,7],[6,2,9],[4,8,1],[7,3,5]]`

If we take a look at the progression of our code execution, we get this:

```
# result of print(i, j, t, decisions, cache.get((i, j)) is not None) on each call to subproblem

0 0 0 [] False
1 1 3 [0] False
2 0 5 [0, 1] False
3 1 9 [0, 1, 0] False
4 0 12 [0, 1, 0, 1] False
4 2 12 [0, 1, 0, 1] False
3 2 9 [0, 1, 0] False
4 0 14 [0, 1, 0, 2] True
```

We follow a path which is locally optimal but globally suboptimal, cache the result, then each subsequent time we reach the same ending we pull the suboptimal value. This is happening in the intermediate steps as well.

The cached suboptimal solution we get is `[0, 1, 0, 1]`. The optimal solution, easily derived from the input, is `[0, 1, 2, 1]`. Based on the order of recursion we're doing, the first solution gets run before the second, and occupies the cache slot. 

But why couldn't we follow the optimal path directly? Naively, that would involve always picking the branch with the lower cost first. But that doesn't always hold, because our paths are influenced both by the current decision being made, but also the equidistance constraint which is only accounted for much later. So we can get forced into a situation where the path cost is always the shortest, but we end up at a house down the line where the preceding house is one color and the equidistant house is another, leaving a single costly option for the current house. Or, as is what happens with the second default testcase on Leetcode for this problem, we end up in a situation where we end up with 2 costly options on the first (cache-miss) pass, and end up picking an option which is globally costlier. 

The solution is to rewrite the subproblem as one which can be solved optimally using data contained within the subproblem only. For this question, that looks like this: "We want the minimum cost for to make houses `0:i` and `n-i-1:n` beautiful simultaneously."

Why does that fulfill the constraint we need to cache? Because, if we have the decision for the `i-1`th and `n-i`th houses as previously available state to calculate the next subproblem's answer, and we use the next subproblem as `i` and `n-i-1` simultaneously, then we can enforce both contraints with optimal solutions on both end simultanously. And this is the essence of dynamic programming, because we can only use it assuming the subproblem we are builing on has the optimal solution already. Let's take a look at the right solution with caching:

In [47]:
def minCost(n: int, cost: List[List[int]]) -> int:
    cache = dict()

    def subproblem(i, lj, rj):
        if i == n // 2:
            return 0
        
        if cache.get((i, lj, rj)):
            return cache[(i, lj, rj)]
        
        r1 = float("inf")

        for (nlj, nrj) in [(0, 1), (0, 2), (1, 2), (2, 1), (2, 0), (1, 0)]:
            if nlj != lj and nrj != rj:
                r1 = min(r1, cost[i][lj] + cost[n - i - 1][rj] + subproblem(i + 1, nlj, nrj))
        
        cache[(i, lj, rj)] = r1

        return r1
            
    result = min([
        subproblem(0, 0, 1),
        subproblem(0, 0, 2),
        subproblem(0, 1, 0),
        subproblem(0, 1, 2),
        subproblem(0, 2, 0),
        subproblem(0, 2, 1),
    ])

    return result

print(minCost(4, [[3,5,7],[6,2,9],[4,8,1],[7,3,5]]))

9


A few things we didn't cover above: 
- how do I index the result to the subproblem (i.e. how do I distinguish each subproblem from the others)? We now need to record the decision on the left and right of centre because they are being made simultaneously. This gives us the 3D shape `(i, lj, rj)` where `lj` is the decision made left of centre and `rj` is made right of centre.
- what action do I take to go to the next subproblem? Now, we need to make the decision for `lj` and `rj` simultaneously, so we try every permutation of 
`(lj, rj)` $\in$ `[(0, 1), (0, 2), (1, 2), (2, 1), (2, 0), (1, 0)]` so that the equidistance constraint is held, and we only use pairs where both values are not equal to the previous one. We also increment `i` like before.

The real question: how does the new subproblem definition avoid this issue? "we end up in a situation where we end up with 2 costly options on the first (cache-miss) pass, and end up picking an option which is globally costlier." The answer is in the recursion:

```
r1 = min(r1, cost[i][lj] + cost[n - i - 1][rj] + subproblem(i + 1, nlj, nrj))
```

From this, we are testing every permutation, along with the entire recursion, such that we are picking a globally optimal path (we consider the total cost of every pair chosen after the current pair). Thus, we will consider all options globally, while using previously decided values as a constraint.

This can be easily unrolled into an iterative solution using the trick used for the edit distance problem. Note the table has a slightly weird shape: we use subtables for the pairs `(i, n - i - 1)` since we attempt to optimize them at the same time.

In [49]:
def minCost(n: int, cost: List[List[int]]) -> int:
    t = [[[float("inf") for _ in range(3)] for _ in range(3)] for _ in range(n // 2)]

    for i in range(n // 2):
        for (nlj, nrj) in [(0, 1), (0, 2), (1, 2), (2, 1), (2, 0), (1, 0)]:
            t[i][nlj][nrj] = cost[i][nlj] + cost[n - i - 1][nrj] + (
                min(
                    [t[i - 1][lj][rj] for (lj, rj) in [(0, 1), (0, 2), (1, 2), (2, 1), (2, 0), (1, 0)] if lj != nlj and rj != nrj]
                ) 
                if i > 0 else 0
            )

    return min([min(x) for x in t[-1]])

print(minCost(4, [[3,5,7],[6,2,9],[4,8,1],[7,3,5]]))

9


Let's look at one more problem.

### Looking ahead

The last problem for today is [regular expression matching](https://leetcode.com/problems/regular-expression-matching/solutions/?envType=problem-list-v2&envId=dynamic-programming). Most of this problem is simple enough, but the tricky part is dealing with the `*` symbols. Luckily, it becomes pretty easy after answering our 3 questions:

1. what subproblem am I solving?

Bottom-up: Does the string up to the current character match the pattern up to the current point?
Top-down: Does the rest of the string match the rest of the pattern?

2. how do I index the result to the subproblem (i.e. how do I distinguish each subproblem from the others)?

Tuple `(i, j)` where `i` is our index in the string, and `j` is our index in the pattern.

3. what action do I take to go to the next subproblem?

We have 3 cases here:
- current character in the string matches the current character in the pattern, or the current character in the pattern is `.`: move to subproblem `(i + 1, j + 1)`
- character after the current one in the pattern is `*`: we need to either match the character preceding the `*` (or any character if that character is `.`), or move to the next part of the expression. That is, either subproblem `(i + 1, j)` (if we decide that the current character is a match we want to accept) or `(i, j + 2)` (if we wish to stop matching the character preceding `*`).

This leads us to a naive recursive implementation, using the top-down subproblem definition:

In [ ]:
def isMatch(s: str, p: str) -> bool:
    def subproblem(i, j):
        if i == len(s):
            cond = j == len(p)
            cond2 = True
            for k in range(j, len(p), 2):
                cond2 &= (k + 1 < len(p) and p[k + 1] == "*")
            return cond or cond2
        elif j == len(p):
            return False
        
        if j + 1 < len(p) and p[j + 1] == "*":
            if s[i] == p[j] or p[j] == ".":
                return subproblem(i + 1, j) or subproblem(i, j + 2)
            else:
                return subproblem(i, j + 2)
        if s[i] == p[j] or p[j] == ".":
            return subproblem(i + 1, j + 1)
        
        return False

    return subproblem(0, 0)

print(isMatch("aa", "a"))
print(isMatch("aa", "a*"))
print(isMatch("ab", ".*"))

False
True
True


Now it turns out the naive recursive implementation is accpetable to leetcode, and in the recursive implementation here there is no way to cache. That means to really scale this solution we would need to switch to an iterative approach. This would also necessitate a switch to the bottom-up problem definition, which works a bit differently. The subproblem becomes, "does the pattern up to `j` match the string up to `i`"? For this, there's a few tricks we need to consider as well.

First off, the empty string `""` matches the empty pattern `""` as well as any patterns with `*` expressions only. Thus, we need to account for this by making our table 1 larger than just the length of the string. We can also process the `*` expressions for the empty string case directly. To do both of these, we use the initial loop from the code below:

```
t = [[False for _ in range(len(p) + 1)] for _ in range(len(s) + 1)]

t[0][0] = True
for j in range(1, len(p) + 1):
    if p[j - 1] == '*':
        t[0][j] = t[0][j - 2]
```

This creates the table, covers the `""` string and `""` pattern case, and for each `*`, sets the result against the empty string to be the same as the result 2 pattern steps back. This is to ensure that if there's a character in the middle without a `*` next to it, it's accounted for. Then, in the main loop, since we've handled the `""` string case, we only need to consider the case where the pattern and string are non-empty. We then need to decrement each of the indices before doing any array accesses. The only remaining caveat is handling `*` against a non-empty string:

```
if p[j - 1] == "*":
    if s[i - 1] == p[j - 2] or p[j - 2] == ".":
        t[i][j] = t[i - 1][j] or t[i][j -21]
    else:
        t[i][j] = t[i][j - 2]
```

This branching logic looks different from what we have above, but it's really similar. First we have a case where the current character matches the chaarcter in the pattern under the `*`. We can either take the value for the previous character in the string `t[i - 1][j]`, or the character before the one affected by the `*`, `t[i][j - 2]`, to say we are finished iterating. Otherwise, we take the character before the one affected by the `*`, `t[i][j - 2]`. Here, `t[i][j - 2]` is analogous to jumping ahead 2 characters (`dfs(i, j + 2)`), and `t[i - 1][j]` is similar to jumping ahead 1 character in the string (`t[i + 1][j]`).

In [50]:
def isMatch(s: str, p: str) -> bool:
    t = [[False for _ in range(len(p) + 1)] for _ in range(len(s) + 1)]

    t[0][0] = True
    for j in range(1, len(p) + 1):
        if p[j - 1] == '*':
            t[0][j] = t[0][j - 2]

    for i in range(1, len(s) + 1):
        for j in range(1, len(p) + 1):
            if p[j - 1] == "*":
                if s[i - 1] == p[j - 2] or p[j - 2] == ".":
                    t[i][j] = t[i - 1][j] or t[i][j - 1]
                else:
                    t[i][j] = t[i][j - 2]
            elif s[i - 1] == p[j - 1] or p[j - 1] == ".":
                t[i][j] = t[i - 1][j - 1]

    return t[-1][-1]

print(isMatch("aa", "a"))
print(isMatch("aa", "a*"))
print(isMatch("ab", ".*"))

False
True
True


And we're done! That's enough leetcode practice for now. I'll get back to this eventually, but hopefully I'm never so bored I have to open leetcode again.